First we are goin to load the data, then split the data.

In [1]:
def extract_L_set(filepath):
    l_set = []
    capture = False
    with open(filepath, 'r') as file:
        for line in file:
            line = line.strip()
        
            # Start capturing when we see the header
            if line.startswith("L set:"):
                capture = True
                continue
        
            # Stop capturing when we reach the B vector
            if line.startswith("B vector:"):
                capture = False
                break
            
            # If we are in the capture zone and the line looks like a list
            if capture and line.startswith("["):
                # ast.literal_eval safely converts the string "[1, 2...]" into a Python list
                row = ast.literal_eval(line)
                l_set.append(row)
    return l_set

In [9]:
import numpy as np
import ast #abstract syntax tree module for safe evaluation of string.
loaded_train_test_data = []
#first we loading the L set from 2003 to 2023
filePath = '../ED_Calculation/2003_2023/results/finalize_L_set_from_2003_to_2023.txt'
loaded_train_test_data = np.array(extract_L_set(filePath))


In [10]:
from sklearn.model_selection import train_test_split
#separate the test_data for Feature and Target
X = loaded_train_test_data[:, :5] #first five col of the row is feature
y = loaded_train_test_data[:, 5] # 6th value is the target

#split for train and val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train {X_train.shape}")
print(f"Shape of y_train{y_train.shape}")
print(f"Shape of X_val{X_val.shape}")
print(f"Shape of y_val{y_val.shape}")

Shape of X_train (116, 5)
Shape of y_train(116,)
Shape of X_val(29, 5)
Shape of y_val(29,)


It is time to define the test data from **2024 to 2025**.

In [17]:
loaded_test_data = []
test_filepath = "../ED_Calculation/2024_current/results/final_result_df_2024_2025.txt"
loaded_test_data = np.array(extract_L_set(test_filepath))
X_test = loaded_test_data[:, :5]
y_test = loaded_test_data[:, 5]


Now we are going to make a Feed Forward Neural Network, which has input layer with 5 feature, a hidden layer with initial neuron size 10 with signmoid function, and a signle neuron output layer with linear function.

In [65]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input


#Create a model
ffnn = Sequential()
# Input layer (5 features)tells the model to expect 5 columns
ffnn.add(Input(shape=(X_train.shape[1],)))
# Hidden layer (10 neurons) 
ffnn.add(Dense(units=10, activation="sigmoid",))
ffnn.add(Dense(units=1, activation="linear"))
ffnn.compile(optimizer="adam", loss='mse', metrics=['mae'])
ffnn.summary()


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 10)             │            60 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71 (284.00 B)

 Trainable params: 71 (284.00 B)

 Non-trainable params: 0 (0.00 B)

Now run and train the model

In [66]:

ffnn_history = ffnn.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), verbose=1,)
print("Model compiled and trained successfully.")

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - loss: 0.9246 - mae: 0.8208 - val_loss: 1.2901 - val_mae: 0.9850
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.8724 - mae: 0.7956 - val_loss: 1.2229 - val_mae: 0.9503
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.8221 - mae: 0.7722 - val_loss: 1.1588 - val_mae: 0.9159
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.7748 - mae: 0.7475 - val_loss: 1.0979 - val_mae: 0.8836
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.7295 - mae: 0.7238 - val_loss: 1.0407 - val_mae: 0.8528
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.6880 - mae: 0.7002 - val_loss: 0.9863 - val_mae: 0.8234
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.6523 - mae: 0.6796 - val_loss: 0.9344 - val_mae: 0.7957
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.6155 - mae: 0.6579 - val_loss: 0.8872 - val_mae: 0.7694
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.5823 - mae: 0.6373 -

In [67]:
test_results = ffnn.evaluate(X_test, y_test, verbose=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 0.0840 - mae: 0.2174


In [68]:
print(f"Test MAE: {test_results[1]:.4}")


Test MAE: 0.2174


In [70]:
import pandas as pd
predictions = ffnn.predict(X_test)

# Create a comparison table
results_df = pd.DataFrame({
    'Actual Value': y_test,
    'Predicted Value': predictions.flatten(),
    'Difference': y_test - predictions.flatten()
})

print("Predictions for the 2024-2025 Test Set:")
display(results_df)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Predictions for the 2024-2025 Test Set:


,Actual Value,Predicted Value,Difference
0,-0.017473,-0.033673,0.016200
1,-0.339443,0.207721,-0.547165
2,0.031541,0.084318,-0.052777
3,-0.215161,-0.140963,-0.074198
4,-0.087107,0.081438,-0.168545
5,0.485383,-0.035807,0.521189
6,-0.220254,0.216264,-0.436519
7,0.307394,0.058590,0.248805
8,0.306878,0.066979,0.239899
9,0.113420,0.038814,0.074606
